In [6]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/processed/olist.db")

In [7]:
query1 = """
SELECT
    p.product_category_name,
    strftime('%Y-%m', o.order_purchase_timestamp) AS month,
    SUM(oi.price) AS revenue,
    COUNT(DISTINCT oi.order_id) AS num_orders
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, month
ORDER BY month, revenue DESC;
"""

category_revenue = pd.read_sql(query1, conn)
category_revenue.head(20)

,product_category_name,month,revenue,num_orders
0,beleza_saude,2016-09,134.97,1
1,moveis_decoracao,2016-10,5690.52,49
2,perfumaria,2016-10,4829.10,26
3,brinquedos,2016-10,4106.50,22
4,consoles_games,2016-10,3619.36,7
5,beleza_saude,2016-10,3439.25,36
6,relogios_presentes,2016-10,2711.07,4
7,esporte_lazer,2016-10,2012.15,17
8,bebes,2016-10,1371.17,9
9,ferramentas_jardim,2016-10,1359.88,5


In [8]:
import os
os.makedirs("../data/processed/query_results", exist_ok=True)
category_revenue.to_csv("../data/processed/query_results/category_revenue_by_month.csv", index=False)

In [9]:
query2 = """
SELECT
    o.order_id,
    r.review_score,
    julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) AS delivery_delay_days
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL;
"""

delivery_review = pd.read_sql(query2, conn)
delivery_review.head(10)

,order_id,review_score,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,4,-7.107488
1,53cdb2fc8bc7dce0b6741e2150273451,4,-5.355729
2,47770eb9100c2d0c44946d9cf07ec65d,5,-17.245498
3,949d5b44dbf5de918fe9c16f97b45f8a,5,-12.980069
4,ad21c59c0840e6cb83a9ceb5573f8159,5,-9.238171
5,a4591c265e18cb1dcee52889e2d8acc3,4,-5.543113
6,6514b8ad8028c9f2cc2374ded245783f,5,-11.461215
7,76c6e866289321a7c93b82b54852dc33,1,-31.410995
8,e69bfb5eb88e0ed6a785585b27e16dbf,5,-6.281597
9,e6ce16cb79ec1d90b1da9085a6118aeb,1,-8.528808


In [10]:
delivery_review["delay_bucket"] = pd.cut(
    delivery_review["delivery_delay_days"],
    bins=[-100, -7, 0, 7, 100],
    labels=["Early (7+ days)", "Early/on-time", "Late (up to 7 days)", "Very late (7+ days)"]
)

delay_summary = delivery_review.groupby("delay_bucket")["review_score"].agg(["mean", "count"])
delay_summary

,mean,count
delay_bucket,,
Early (7+ days),4.316477,71329
Early/on-time,4.199896,17319
Late (up to 7 days),3.183830,4428
Very late (7+ days),1.715832,3234


In [12]:
delivery_review.to_csv("../data/processed/query_results/delivery_delay_vs_review.csv", index=False)
delay_summary.to_csv("../data/processed/query_results/delay_bucket_summary.csv")

In [13]:
query3 = """
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS num_orders,
    SUM(oi.price) AS total_spent
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id
ORDER BY num_orders DESC;
"""

customer_orders = pd.read_sql(query3, conn)
customer_orders.head(10)

,customer_unique_id,num_orders,total_spent
0,8d50f5eadf50201ccdcedfb9e2ac8455,15,714.63
1,3e43e6105506432c953e165fb2acf44c,9,1000.85
2,ca77025e7201e3b30c44b472ff346268,7,806.61
3,6469f99c1f9dfae7733b25662e7f1782,7,664.20
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7,809.21
5,f0e310a6839dce9de1638e0fe5ab282a,6,438.09
6,dc813062e0fc23409cd255f7f53c7074,6,687.76
7,63cfc61cee11cbe306bff5857d00bfe4,6,579.81
8,47c1a3033b8b77b3ab6e109eb4d5fdf3,6,836.60
9,12f5d6e1cbf93dafd9dcc19095df0b3d,6,58.40


In [14]:
customer_orders["customer_type"] = customer_orders["num_orders"].apply(
    lambda x: "Repeat buyer" if x > 1 else "One-time buyer"
)

repeat_summary = customer_orders["customer_type"].value_counts()
repeat_summary_pct = customer_orders["customer_type"].value_counts(normalize=True) * 100

print(repeat_summary)
print(repeat_summary_pct)

customer_type
One-time buyer    90557
Repeat buyer       2801
Name: count, dtype: int64
customer_type
One-time buyer    96.999722
Repeat buyer       3.000278
Name: proportion, dtype: float64


In [15]:
customer_orders.to_csv("../data/processed/query_results/customer_order_counts.csv", index=False)

In [16]:
query4 = """
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS num_orders,
    SUM(oi.price) AS total_revenue,
    AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)) AS avg_delivery_days
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY total_revenue DESC;
"""

state_summary = pd.read_sql(query4, conn)
state_summary

,customer_state,num_orders,total_revenue,avg_delivery_days
0,SP,40494,5066562.98,8.723845
1,RJ,12350,1759651.13,15.147896
2,MG,11354,1552481.83,11.982504
3,RS,5344,728718.47,15.188566
4,PR,4923,666063.51,11.947217
5,SC,3546,507012.13,14.997643
6,BA,3256,493584.14,19.247618
7,DF,2080,296498.41,12.958993
8,GO,1957,282836.70,15.403338
9,ES,1995,268643.45,15.649066


In [17]:
state_summary.to_csv("../data/processed/query_results/revenue_delivery_by_state.csv", index=False)

In [18]:
state_summary.sort_values("avg_delivery_days").head(5)  # fastest 5

,customer_state,num_orders,total_revenue,avg_delivery_days
0,SP,40494,5066562.98,8.723845
4,PR,4923,666063.51,11.947217
2,MG,11354,1552481.83,11.982504
7,DF,2080,296498.41,12.958993
5,SC,3546,507012.13,14.997643


In [19]:
state_summary.sort_values("avg_delivery_days").tail(5)  # slowest 5

,customer_state,num_orders,total_revenue,avg_delivery_days
12,PA,946,174470.59,23.754053
19,AL,397,78855.72,24.489142
23,AM,145,22155.84,26.390967
25,AP,67,13374.81,28.205960
26,RR,41,7057.47,28.231011
